# [12.1] CLIP, SigLIP, and VLM Controls

> **ARENA extension note.** Original ARENA content is unchanged. This appended section starts the VLM ladder with a tiny controlled CLIP before it escalates to pinned real-model CUDA evidence.

By the end of this notebook, you will have shown that a tiny CLIP trained on controlled colored-shape image/caption pairs learns a shared image-text embedding space, because paired retrieval succeeds while random-caption and image/text conflict controls fail.

## Core Question

When a vision-language model says an image and a caption match, what result would make that hard to fake?

For this notebook, the answer is a visible result with controls: a generated image grid, a falling contrastive loss curve, a retrieval heatmap with a strong diagonal, top-k rows showing the exact captions retrieved, and two negative controls that break the claim.



**Common bug:** treating a pretty diagonal or plausible top caption as enough evidence. In this notebook, the claim only passes when matched retrieval succeeds and the random-caption plus wrong-color controls fail.

## Learning Objectives

By the end, you should be able to:

1. implement normalized CLIP-style image-text logits;
2. evaluate bidirectional retrieval and positive-pair margins;
3. implement the SigLIP pairwise logistic objective;
4. train a tiny linear CLIP on rendered colored-shape images;
5. build random-caption and counterfactual-caption controls;
6. interpret a retrieval heatmap and top-caption table; and
7. read the committed real-model CUDA report without mistaking it for the lesson.


In [ ]:
GT_TIER = "GT-1"
EXERCISE_ID = "12_1_clip_siglip_and_vlm_controls"
DIFFICULTY = 4
IMPORTANCE = 3
EXPECTED_RUNTIME = "seconds on toy contract; minutes on local real-model path"
REQUIRES_GPU = True  # exercises are CPU-friendly; the committed real-model report is CUDA-backed


## Setup

The notebook keeps the important operations in front of you. Boring rendering and data-schema helpers live in `arena_ext.vlm_interpretability`; the CLIP scoring, SigLIP loss, tiny training loop, retrieval table, and controls are the exercises.


In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch as t
import torch.nn.functional as F
from IPython.display import display

chapter = "chapter12_vlm_interpretability"
section = "part1_clip_siglip_vlm_controls"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part1_clip_siglip_vlm_controls.tests as tests
from arena_ext.vlm_interpretability import (
    ContrastiveAlignmentReport,
    ToyCLIPTrainingResult,
    VisualTokenAttributionReport,
    build_toy_clip_batch,
    deterministic_derangement,
    retrieval_accuracy,
    retrieval_table,
    toy_caption_features,
)


## Cold Open: The Data Should Be Inspectable

Before any model claim, look at the actual examples. These are not web images and not a hidden benchmark. They are controlled colored shapes where the true caption, counterfactual caption, and misleading text prior are all known.


In [ ]:
toy_batch = build_toy_clip_batch(
    colors=("red", "blue", "green", "yellow"),
    shapes=("square", "circle", "triangle"),
    image_size=48,
)

fig, axes = plt.subplots(3, 4, figsize=(9, 6))
for ax, image, caption in zip(axes.flat, toy_batch.image_tensors, toy_batch.captions):
    ax.imshow(image.permute(1, 2, 0).numpy())
    ax.set_title(caption, fontsize=9)
    ax.axis("off")
fig.suptitle("Toy CLIP image grid: every claim below refers to these examples", y=0.98)
plt.tight_layout()
plt.show()

display(pd.DataFrame([
    {
        "image_id": scene.image_id,
        "caption": caption,
        "counterfactual_caption": f"a {scene.counterfactual_answer} {scene.shape}",
        "question": scene.question,
    }
    for scene, caption in zip(toy_batch.scenes, toy_batch.captions)
]).head(8))


### Exercise - CLIP-Style Retrieval Logits

> Difficulty: medium
> Importance: high
>
> You should spend 10 minutes on this exercise.

CLIP normalizes image and text embeddings, takes all pairwise dot products, and multiplies by a positive logit scale. Implement the score and a report that checks both image-to-text and text-to-image retrieval.


In [ ]:
def _l2_normalize(values: t.Tensor, *, eps: float = 1e-8) -> t.Tensor:
    return values.float() / values.float().norm(dim=-1, keepdim=True).clamp_min(eps)


def clip_contrastive_logits(
    image_embeddings: t.Tensor,
    text_embeddings: t.Tensor,
    *,
    logit_scale: float = 10.0,
) -> t.Tensor:
    raise NotImplementedError()


def contrastive_alignment_report(
    logits: t.Tensor,
    *,
    min_accuracy: float = 1.0,
    min_positive_margin: float = 1.0,
) -> ContrastiveAlignmentReport:
    raise NotImplementedError()


def contrastive_smoke_test() -> dict:
    image_embeddings = t.eye(3)
    text_embeddings = t.eye(3)
    logits = clip_contrastive_logits(image_embeddings, text_embeddings, logit_scale=5.0)
    return contrastive_alignment_report(
        logits,
        min_accuracy=1.0,
        min_positive_margin=4.0,
    ).__dict__


tests.test_contrastive_smoke_test(contrastive_smoke_test)


<details>
<summary>Expected output</summary>

```text
All tests in `test_contrastive_smoke_test` passed!
```

The identity example should have diagonal logits of `5.0`, strongest off-diagonal logits of `0.0`, and mean positive margin `5.0`.

</details>

<details>
<summary>Help - check the reasoning before opening the solution</summary>

Accuracy asks whether the correct caption wins. The positive-pair margin asks whether it wins against the strongest distractor in each direction. CLIP retrieval demos often look impressive until you add close captions; this margin is the first guardrail.

</details>

<details>
<summary>Solution</summary>

The solution normalizes both embedding matrices, computes `logit_scale * image @ text.T`, then compares the diagonal against the strongest off-diagonal image and text distractors.

</details>


### Exercise - SigLIP Pairwise Loss

CLIP uses a softmax over the current batch. SigLIP instead treats every image-text pair as a binary classification example. Matching pairs get label `+1`; mismatched pairs get label `-1`.


In [ ]:
def siglip_pairwise_loss(logits: t.Tensor, labels: t.Tensor) -> t.Tensor:
    raise NotImplementedError()


def siglip_smoke_test() -> dict:
    logits = t.tensor([[4.0, -4.0], [-3.0, 3.0]])
    labels = t.eye(2)
    return {"loss": siglip_pairwise_loss(logits, labels).item()}


tests.test_siglip_smoke_test(siglip_smoke_test)


<details>
<summary>Expected output</summary>

```text
All tests in `test_siglip_smoke_test` passed!
```

The loss should be about `0.03337` because all four signed margins are confident and correct.

</details>

<details>
<summary>Help - check the reasoning before opening the solution</summary>

Convert labels to `+1` and `-1`, multiply by the logits, and use `softplus(-signed_margin)`. A confident correct positive pair and a confident correct negative pair both have small loss.

</details>

<details>
<summary>Solution</summary>

Use `signed_labels = torch.where(labels > 0, 1.0, -1.0)` and average `F.softplus(-signed_labels * logits)`.

</details>


### Exercise - Train a Tiny CLIP

Now build the thing. The image features are simple pixel statistics from the rendered image grid; the text features are color/shape bag-of-words vectors. Your job is to learn two linear projections into a shared embedding space using the CLIP loss from Exercise 1.


In [ ]:
def train_toy_clip_projectors(
    image_features: t.Tensor,
    text_features: t.Tensor,
    *,
    embedding_dim: int = 8,
    steps: int = 250,
    lr: float = 0.05,
    logit_scale: float = 10.0,
    seed: int = 0,
) -> ToyCLIPTrainingResult:
    raise NotImplementedError()


trained = train_toy_clip_projectors(toy_batch.image_features, toy_batch.text_features)
assert trained.train_losses[-1] < 0.1 * trained.train_losses[0]
assert trained.report.aligned

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(trained.train_losses)
ax.set_xlabel("step")
ax.set_ylabel("CLIP loss")
ax.set_title("Tiny CLIP learns the paired image-text space")
plt.show()

print({
    "loss_start": round(trained.train_losses[0], 4),
    "loss_end": round(trained.train_losses[-1], 4),
    "image_to_text_accuracy": trained.report.image_to_text_accuracy,
    "text_to_image_accuracy": trained.report.text_to_image_accuracy,
    "mean_positive_margin": round(trained.report.mean_positive_margin, 3),
})


<details>
<summary>Expected output</summary>

You should see a smooth loss curve falling from roughly `4.08` to below `0.05`. The printed retrieval summary should have both accuracies equal to `1.0` and a mean positive margin above `1.0`.

</details>

<details>
<summary>Help - check the reasoning before opening the solution</summary>

Train two matrices: one maps image features to the shared embedding space, and the other maps text features there. Each step should compute the full image-text logit matrix and average image-to-text and text-to-image cross entropy.

</details>

<details>
<summary>Solution</summary>

The solution uses `torch.nn.Parameter` matrices, `torch.optim.Adam`, the CLIP logits from Exercise 1, and symmetric cross entropy over the diagonal targets.

</details>


### Exercise - Retrieval Heatmap and Top-Caption Rows

A good VLM lesson should show the examples, not just a scalar. Turn the retrieval matrix into a heatmap and a table of the top retrieved caption for each image.


In [ ]:
def top_retrieval_rows(
    logits: t.Tensor,
    captions: tuple[str, ...],
    image_ids: tuple[str, ...],
) -> list[dict[str, float | int | str]]:
    raise NotImplementedError()


image_ids = tuple(scene.image_id for scene in toy_batch.scenes)
rows = top_retrieval_rows(trained.retrieval_logits, toy_batch.captions, image_ids)
assert all(row["target_rank"] == 1 for row in rows)
display(pd.DataFrame(rows))

fig, ax = plt.subplots(figsize=(7, 6))
heatmap = ax.imshow(trained.retrieval_logits.numpy(), cmap="viridis")
ax.set_xticks(range(len(toy_batch.captions)), toy_batch.captions, rotation=90, fontsize=7)
ax.set_yticks(range(len(image_ids)), image_ids, fontsize=7)
ax.set_title("Tiny CLIP retrieval logits: correct pairs should form the diagonal")
fig.colorbar(heatmap, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()


<details>
<summary>Expected output</summary>

Every row in the table should have `target_rank == 1`. The heatmap should show a bright diagonal rather than a uniform block or random speckle.

</details>

<details>
<summary>Help - check the reasoning before opening the solution</summary>

For each image row, sort captions by descending logit. The rank is the position of the diagonal caption in that sorted order plus one.

</details>

<details>
<summary>Solution</summary>

Use `argsort(descending=True)` on each row, then find where the row index appears in the sorted caption indices.

</details>


### Exercise - Controls That Should Fail

The paired heatmap is only meaningful if nearby bad explanations fail. We use two controls:

1. a deranged-caption control, where every caption is assigned to a different image; and
2. a counterfactual color-caption control, where the shape is right but the color is deliberately wrong.


In [ ]:
def caption_derangement_control(
    image_embeddings: t.Tensor,
    text_embeddings: t.Tensor,
    *,
    seed: int = 0,
) -> tuple[t.Tensor, t.Tensor]:
    raise NotImplementedError()


random_logits, random_permutation = caption_derangement_control(
    trained.image_embeddings,
    trained.text_embeddings,
    seed=0,
)
random_report = contrastive_alignment_report(random_logits, min_accuracy=1.0, min_positive_margin=1.0)

conflict_captions = tuple(
    f"a {scene.counterfactual_answer} {scene.shape}" for scene in toy_batch.scenes
)
conflict_features = toy_caption_features(
    conflict_captions,
    colors=toy_batch.colors,
    shapes=toy_batch.shapes,
)
conflict_embeddings = conflict_features @ trained.text_projection
conflict_logits = clip_contrastive_logits(trained.image_embeddings, conflict_embeddings)
conflict_report = contrastive_alignment_report(conflict_logits, min_accuracy=1.0, min_positive_margin=1.0)

control_summary = pd.DataFrame([
    {
        "condition": "paired captions",
        "image_to_text_accuracy": trained.report.image_to_text_accuracy,
        "mean_margin": trained.report.mean_positive_margin,
        "aligned": trained.report.aligned,
    },
    {
        "condition": "deranged captions",
        "image_to_text_accuracy": retrieval_accuracy(random_logits),
        "mean_margin": random_report.mean_positive_margin,
        "aligned": random_report.aligned,
    },
    {
        "condition": "wrong color captions",
        "image_to_text_accuracy": retrieval_accuracy(conflict_logits),
        "mean_margin": conflict_report.mean_positive_margin,
        "aligned": conflict_report.aligned,
    },
])
display(control_summary)
assert trained.report.aligned and not random_report.aligned and not conflict_report.aligned


<details>
<summary>Expected output</summary>

The paired row should have accuracy `1.0` and `aligned == True`. The deranged-caption and wrong-color rows should have low accuracy and `aligned == False`.

</details>

<details>
<summary>Help - check the reasoning before opening the solution</summary>

A control is only useful if it is matched in shape to the real result. Deranging captions keeps the same images, captions, and model; it only breaks the pairing.

</details>

<details>
<summary>Solution</summary>

Use a seeded derangement, index the text embeddings by that permutation, recompute CLIP logits, and evaluate the same report.

</details>


### Exercise - Tiny Visual-Token Attribution

This is not yet a real VLM patching lesson. It is the locality check the later VLM notebooks will reuse: if a text direction is about the object, most positive attribution mass should sit on object-like tokens rather than background-like tokens.


In [ ]:
def visual_token_attribution_report(
    token_activations: t.Tensor,
    text_direction: t.Tensor,
    *,
    top_k: int = 2,
    min_top_token_mass: float = 0.6,
) -> VisualTokenAttributionReport:
    raise NotImplementedError()


def token_attribution_smoke_test() -> dict:
    token_activations = t.tensor(
        [
            [0.0, 0.0],
            [3.0, 0.0],
            [2.0, 0.0],
            [0.0, 1.0],
        ]
    )
    text_direction = t.tensor([1.0, 0.0])
    report = visual_token_attribution_report(
        token_activations,
        text_direction,
        top_k=2,
        min_top_token_mass=0.8,
    )
    result = report.__dict__.copy()
    result["token_scores"] = report.token_scores.tolist()
    result["top_token_indices"] = report.top_token_indices.tolist()
    return result


tests.test_token_attribution_smoke_test(token_attribution_smoke_test)


<details>
<summary>Expected output</summary>

```text
All tests in `test_token_attribution_smoke_test` passed!
```

The toy token scores should be `[0.0, 3.0, 2.0, 0.0]`, with top tokens `[1, 2]` and top-token mass `1.0`.

</details>

<details>
<summary>Help - check the reasoning before opening the solution</summary>

Project every token activation onto the normalized text direction. Then ask how much of the positive mass is captured by the top `k` tokens.

</details>

<details>
<summary>Solution</summary>

Normalize the direction, compute dot products, take `topk`, and divide positive mass on those top tokens by total positive mass.

</details>


## Signature Result

The signature result is the visible learner artifact: image grid, loss curve, retrieval heatmap, top-caption table, and failed controls. The committed real-model report is supporting evidence, not a replacement for this result.


In [ ]:
paired_acc = retrieval_accuracy(trained.retrieval_logits)
random_acc = retrieval_accuracy(random_logits)
conflict_acc = retrieval_accuracy(conflict_logits)

signature_summary = pd.DataFrame([
    {"result": "paired retrieval", "value": paired_acc, "passes": paired_acc == 1.0},
    {"result": "random-caption control", "value": random_acc, "passes": random_acc <= 0.25},
    {"result": "wrong-color control", "value": conflict_acc, "passes": conflict_acc <= 0.25},
    {"result": "final loss", "value": trained.train_losses[-1], "passes": trained.train_losses[-1] < 0.1},
])
display(signature_summary)

assert paired_acc == 1.0
assert random_acc <= 0.25
assert conflict_acc <= 0.25
assert trained.train_losses[-1] < 0.1


## Try It Yourself

Change one input and rerun the small experiment. Good things to try:

- replace `yellow` with `purple`;
- remove `triangle` and see whether the heatmap gets easier;
- increase `seed` and check whether the same controls still fail;
- change a caption to the wrong color and inspect which image it retrieves.


In [ ]:
play_batch = build_toy_clip_batch(
    colors=("red", "blue", "purple"),
    shapes=("square", "circle"),
    image_size=48,
)
play_trained = train_toy_clip_projectors(
    play_batch.image_features,
    play_batch.text_features,
    seed=3,
)
play_rows = retrieval_table(
    play_trained.retrieval_logits,
    play_batch.captions,
    image_ids=tuple(scene.image_id for scene in play_batch.scenes),
)
display(pd.DataFrame(play_rows))
print({
    "paired_accuracy": retrieval_accuracy(play_trained.retrieval_logits),
    "final_loss": round(play_trained.train_losses[-1], 4),
})


## Real-Model CUDA Evidence

The real-model checks live in `verification_report.json` and are regenerated with `scripts/run_extension_verification_reports.py --section 12.1`. They test pinned CLIP, SigLIP, and Qwen2.5-VL paths on deterministic rendered-shape examples, object/background/random region controls, and visual-token activation patching.

This report is necessary for review, but it is not the pedagogical core of the notebook.


In [ ]:
report = json.loads((section_dir / "verification_report.json").read_text())
gpu = report["metrics"]["gpu_test"]
report_summary = pd.DataFrame([
    {"check": "CUDA available", "value": gpu["cuda_available"]},
    {"check": "Toy CLIP control claim", "value": gpu.get("toy_clip_control_claim_passed", "not in old report")},
    {"check": "Real CLIP rendered-shape preflight", "value": gpu["real_clip_rendered_shape_preflight_passed"]},
    {"check": "Real SigLIP rendered-shape preflight", "value": gpu["real_siglip_rendered_shape_preflight_passed"]},
    {"check": "Qwen2.5-VL rendered-shape answers", "value": gpu["real_qwen25_vl_answers"]},
    {"check": "Peak VRAM GB", "value": round(gpu["peak_vram_gb"], 3)},
])
display(report_summary)

def run_smoke_test(cpu: bool = True) -> dict:
    from part1_clip_siglip_vlm_controls import solutions as reference_solutions
    return reference_solutions.run_smoke_test(cpu=cpu)


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    from part1_clip_siglip_vlm_controls import solutions as reference_solutions
    return reference_solutions.run_gpu_test(max_vram_gb=max_vram_gb)


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    from part1_clip_siglip_vlm_controls import solutions as reference_solutions
    return reference_solutions.run_full_experiment(max_vram_gb=max_vram_gb)


tests.test_toy_clip_signature_result_has_controls()
tests.test_clip_siglip_core_notebook_contract(run_smoke_test)
tests.test_committed_verification_report_real_model_controls()


## Limitations and Interpreting the Result

The toy result is strong because the diagonal survives exact controls: the same model, same examples, and same captions fail when the pairing is broken or the color is counterfactual. That is the minimum standard before later notebooks make larger VLM claims.

The toy result is also limited. It does not prove that web-scale CLIP learned human concepts, and it does not make a broad claim about real VLM reasoning. The real CUDA report only supports its pinned rendered-shape claim. Later VLM notebooks should extend this into feature geometry, mini VLM training, visual-token flow, hallucination, and multimodal SAEs.

## Bonus: Anomaly Hunting

If a run produces a weak diagonal or a control that passes, do not polish the plot. Treat it as a negative result. Check the caption permutation, the feature extraction, whether captions name exactly one color and shape, and whether the heatmap is dominated by shape while ignoring color.
